#### Load Data

In [102]:
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.ticker import MultipleLocator
from itertools import product
from openpyxl import Workbook

# load dataset
df = pd.read_csv("../data/coffee_shop_sales.csv")

# display information
print("Shape:", df.shape)
print("\nColumns:")
print(df.columns.tolist())


Shape: (149116, 11)

Columns:
['Transaction ID', 'Transaction Date', 'Transaction Time', 'Transaction Quantity', 'Store ID', 'Store Location', 'Product ID', 'Unit Price', 'Product Category', 'Product Type', 'Product Detail']


#### Add Quarter Columns

In [103]:
df["Transaction Date"] = pd.to_datetime(df["Transaction Date"])

/var/folders/53/vr8gs__x60sg207tybh25bv80000gn/T/ipykernel_81889/3594840833.py:1: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df["Transaction Date"] = pd.to_datetime(df["Transaction Date"])


In [104]:
df["Quarter"] = df["Transaction Date"].dt.to_period("Q")

print(df["Quarter"].value_counts().sort_index())

Quarter
2023Q1    54902
2023Q2    94214
Freq: Q-DEC, Name: count, dtype: int64


In [105]:
for quarter in df["Quarter"].sort_values().unique():
    quarter_df = df[df["Quarter"] == quarter]

    print(
        quarter,
        quarter_df["Transaction Date"].min(),
        quarter_df["Transaction Date"].max(),
        len(quarter_df)
    )

2023Q1 2023-01-01 00:00:00 2023-03-31 00:00:00 54902
2023Q2 2023-04-01 00:00:00 2023-06-30 00:00:00 94214


#### Calculate Business Metrics

In [109]:
quarterly_results = {}

quarters = df["Quarter"].sort_values().unique()

for quarter in quarters:

    # filter data per quarter
    quarter_df = df[df["Quarter"] == quarter].copy()

    print(f"\n===== {quarter} =====")

    # metrics
    quarter_df["Revenue"] = (
    quarter_df["Transaction Quantity"] *
    quarter_df["Unit Price"]
    )
    #------------------------------------------
    total_revenue = quarter_df["Revenue"].sum()

    total_transactions = quarter_df["Transaction ID"].nunique()

    total_units = quarter_df["Transaction Quantity"].sum()

    average_transaction_value = (
        total_revenue / total_transactions
    )

    average_units_per_transaction = (
        total_units / total_transactions
    )
    # print(f"\nMetrics for {quarter}:")
    # print("Revenue:", total_revenue)
    # print("Transactions:", total_transactions)
    # print("Units:", total_units)
    # print("Average Transaction Value:", average_transaction_value)
    # print("Average Units per Transaction:", average_units_per_transaction)


    # product related metrics

    product_sales = (
        quarter_df.groupby("Product Type")["Revenue"]
        .sum()
        .sort_values(ascending=False)
    )

    category_sales = (
        quarter_df.groupby("Product Category")["Revenue"]
        .sum()
        .sort_values(ascending=False)
    )

    quarter_df["Revenue %"] = (
        quarter_df["Revenue"] /
        quarter_df["Revenue"].sum()
    )


    # metrics by product

    product_sales_metrics = (
        quarter_df.groupby("Product Detail")
        .agg(
            revenue=("Revenue", "sum"),
            units_sold=("Transaction Quantity", "sum"),
            transactions=("Transaction ID", "nunique")
        )
        .sort_values("revenue", ascending=False)
    )


    # revenue divided into various time periods

    quarter_df["Transaction Date"] = pd.to_datetime(
        quarter_df["Transaction Date"],
        errors="coerce"
    )

    sales_by_day = (
        quarter_df.groupby("Transaction Date")["Revenue"]
        .sum()
    )

    quarter_df["Transaction Time"] = pd.to_datetime(
        quarter_df["Transaction Time"],
        format="%H:%M:%S",
        errors="coerce"
    )


    # extract hour

    quarter_df["Transaction Hour"] = (
        quarter_df["Transaction Time"].dt.hour
    )


    # sales by hour

    hourly_sales = (
        quarter_df.groupby("Transaction Hour")["Revenue"]
        .sum()
    )

    sales_by_hour = (
        quarter_df.groupby("Transaction Hour")
        .agg(
            revenue=("Revenue", "sum"),
            transactions=("Transaction ID", "nunique"),
            units_sold=("Transaction Quantity", "sum")
        )
        .sort_index()
    )

    peak_hour = hourly_sales.idxmax()


    # sales by day of week

    quarter_df["Day of Week"] = (
        quarter_df["Transaction Date"].dt.day_name()
    )

    day_order = [
        "Monday",
        "Tuesday",
        "Wednesday",
        "Thursday",
        "Friday",
        "Saturday",
        "Sunday"
    ]

    sales_by_weekday = (
        quarter_df.groupby("Day of Week")["Revenue"]
        .sum()
        .reindex(day_order)
    )


    # sales by time period

    def time_period(hour):
        if 6 <= hour < 10:
            return "Morning"
        elif 11 <= hour < 13:
            return "Lunch"
        elif 14 <= hour < 16:
            return "Afternoon"
        else:
            return "Evening"


    quarter_df["Time Period"] = (
        quarter_df["Transaction Hour"].apply(time_period)
    )

    time_order = [
        "Morning",
        "Lunch",
        "Afternoon",
        "Evening"
    ]

    quarter_df["Time Period"] = pd.Categorical(
        quarter_df["Time Period"],
        categories=time_order,
        ordered=True
    )

    sales_by_time_of_day = (
        quarter_df.groupby("Time Period")["Revenue"]
        .sum()
    )


    # revenue by product category and time period

    sales_by_period_and_category = (
        quarter_df.groupby(
            ["Time Period", "Product Category"]
        )["Revenue"]
        .sum()
        .unstack(fill_value=0)
    )


    # location related metrics

    location_sales = (
        quarter_df.groupby("Store Location")["Revenue"]
        .sum()
        .sort_values(ascending=False)
    )

    store_sales = (
        quarter_df.groupby(["Store ID", "Store Location"])
        .agg(
            revenue=("Revenue", "sum"),
            units_sold=("Transaction Quantity", "sum"),
            transactions=("Transaction ID", "nunique")
        )
        .sort_values("revenue", ascending=False)
    )


    # heatmap data

    revenue_by_day_hour = {}

    for location in quarter_df["Store Location"].unique():

        revenue_by_day_hour[location] = (
            quarter_df[
                quarter_df["Store Location"] == location
            ]
            .groupby(
                ["Day of Week", "Transaction Hour"]
            )["Revenue"]
            .sum()
            .unstack(fill_value=0)
            .reindex(day_order)
        )


    # store all results for this quarter
    quarterly_results[quarter] = {

        "total_revenue": total_revenue,
        "total_transactions": total_transactions,
        "total_units": total_units,
        "average_transaction_value": average_transaction_value,
        "average_units_per_transaction": average_units_per_transaction,

        "product_sales": product_sales,
        "category_sales": category_sales,
        "product_detail_sales": product_sales_metrics,

        "sales_by_day": sales_by_day,
        "sales_by_hour": sales_by_hour,
        "sales_by_weekday": sales_by_weekday,
        "sales_by_time_of_day": sales_by_time_of_day,
        "sales_by_period_and_category": sales_by_period_and_category,

        "location_sales": location_sales,
        "store_sales": store_sales,
        "revenue_by_day_hour": revenue_by_day_hour
    }


===== 2023Q1 =====

===== 2023Q2 =====


In [ ]:
# quarterly_results[pd.Period("2023Q1")]["category_sales"]

Product Category
Coffee                98829.40
Tea                   72266.00
Bakery                30477.15
Drinking Chocolate    26723.50
Coffee beans          14578.95
Branded                4926.00
Loose Tea              4218.65
Flavours               3076.80
Packaged Chocolate     1561.16
Name: Revenue, dtype: float64

#### Automation

In [108]:
from csv import writer

from matplotlib.pyplot import bar
from openpyxl.styles import Font, Alignment
import datetime


for quarter, results in quarterly_results.items():

    wb = Workbook()

    # formatting
    header_18 = Font(name='Calibri', size=18, bold=True, color="222222")
    header_14 = Font(name='Calibri', size=14, bold=True, color="222222")

    body_bold = Font(name='Calibri', size=11, bold=True, color="222222")
    body_regular = Font(name='Calibri', size=11, bold=False, color="222222")

    center_align = Alignment(horizontal='center', vertical='center')


    # ####### first sheet - KPIs #######
    kpi_sheet = wb.active

    # name it KPIs
    kpi_sheet.title = "KPIs"

    # set columns width
    kpi_sheet.column_dimensions['A'].width = 20
    kpi_sheet.column_dimensions['B'].width = 20
    kpi_sheet.column_dimensions['C'].width = 20
    kpi_sheet.column_dimensions['D'].width = 20
    kpi_sheet.column_dimensions['E'].width = 20

    # add todays date up top
    kpi_sheet['A1'] = datetime.datetime.now().strftime("%d %B, %Y")
    kpi_sheet.row_dimensions[1].height = 40

    # set header
    kpi_sheet['A2'] = "Sales Performance Report"
    kpi_sheet['A2'].font = Font(name='Calibri', size=28, bold=True, color="222222")
    kpi_sheet.row_dimensions[2].height = 60
    kpi_sheet.merge_cells('A2:E2')
    kpi_sheet['A2'].alignment = center_align

    # set subheader
    start_month = quarter.start_time.strftime("%B")
    end_month = quarter.end_time.strftime("%B")
    year = quarter.start_time.year
    #-----------------------------------
    kpi_sheet["A3"] = f"{start_month} – {end_month} {year}"
    kpi_sheet['A3'].font = header_18
    kpi_sheet.row_dimensions[3].height = 36
    kpi_sheet.merge_cells('A3:E3')
    kpi_sheet['A3'].alignment = center_align

    # blank row
    kpi_sheet["A4"] = ""
    kpi_sheet.row_dimensions[4].height = 30

    # FIRST ROW OF KPIs
    # headers
    kpi_sheet.row_dimensions[5].height = 20
    kpi_sheet["A5"] = "TOTAL REVENUE"
    kpi_sheet["C5"] = "TRANSACTIONS"
    kpi_sheet["E5"] = "AVG TRANSACTION"
    for cell in kpi_sheet[5]:
        cell.alignment = center_align 
        cell.font = header_14

    # numbers
    kpi_sheet.row_dimensions[6].height = 20
    kpi_sheet["A6"] = f"${total_revenue:,.2f}"
    kpi_sheet["C6"] = total_transactions
    kpi_sheet["E6"] = f"${average_transaction_value:,.2f}"
    for cell in kpi_sheet[6]:
        cell.alignment = center_align
        cell.font = body_regular

    # SECOND ROW OF KPIs
    # blank row
    kpi_sheet["A7"] = ""
    kpi_sheet.row_dimensions[7].height = 30

    # headers
    kpi_sheet.row_dimensions[8].height = 20
    kpi_sheet["A8"] = "BEST LOCATION"
    kpi_sheet["C8"] = "BEST PRODUCT "
    kpi_sheet["E8"] = "PEAK PERIOD"
    for cell in kpi_sheet[8]:
        cell.alignment = center_align 
        cell.font = header_14

    # numbers
    kpi_sheet.row_dimensions[9].height = 20
    kpi_sheet["A9"] = store_sales.index[0][1]  # best location
    kpi_sheet["C9"] = product_sales.index[0]  # best product
    kpi_sheet["E9"] = f"{peak_hour}:00 - {peak_hour + 1}:00"  # peak period

    for cell in kpi_sheet[9]:
        cell.alignment = center_align
        cell.font = body_regular

    # ######## SECOND SHEET - SALES ANALYSIS #######
    # sales_analysis = wb.create_sheet(title="Sales Analysis")

    # # set columns width
    # sales_analysis.column_dimensions['A'].width = 20
    # sales_analysis.column_dimensions['B'].width = 20
    # sales_analysis.column_dimensions['C'].width = 20
    # sales_analysis.column_dimensions['D'].width = 20
    # sales_analysis.column_dimensions['E'].width = 20

    # # set header
    # sales_analysis['A1'] = "Sales Analysis"
    # sales_analysis['A1'].font = header_18
    # sales_analysis.row_dimensions[1].height = 60
    # sales_analysis.merge_cells('A1:E1')
    # sales_analysis['A1'].alignment = center_align

    # # set subheader
    # sales_analysis['A2'] = "Revenue performance across products and categories"
    # sales_analysis['A2'].font = header_14
    # sales_analysis.row_dimensions[2].height = 36
    # sales_analysis.merge_cells('A2:E2')
    # sales_analysis['A2'].alignment = center_align

    # # blank row
    # sales_analysis["A3"] = ""
    # sales_analysis.row_dimensions[3].height = 30

    # sales_analysis['A4'] = "Revenue by Product Category"
    # sales_analysis['A4'].font = header_14
    # sales_analysis.row_dimensions[4].height = 36

    # # revenue by product category
    # from openpyxl.utils.dataframe import dataframe_to_rows
    # from openpyxl.chart import BarChart, Reference

    # # create chart from ws
    # chart = BarChart()
    # chart.type = "bar"
    # # chart.title = "Revenue by Product Category"

    # # create hidden worksheet for chart data
    # sales_cat_ws = wb.create_sheet("Revenue by Category Data")

    # # df to excel
    # for row in dataframe_to_rows(
    #     category_sales.reset_index(),
    #     index=False,
    #     header=True
    # ):
    #     sales_cat_ws.append(row)

    # data = Reference(
    #     sales_cat_ws,
    #     min_col=2,
    #     min_row=1,
    #     max_row=len(category_sales) + 1
    # )

    # categories = Reference(
    #     sales_cat_ws,
    #     min_col=1,
    #     min_row=2,
    #     max_row=len(category_sales) + 1
    # )

    # chart.add_data(data, titles_from_data=True)
    # chart.set_categories(categories)

    # chart.height = 6
    # sales_analysis.row_dimensions[5].height = chart.height * 28.35

    # sales_analysis.add_chart(chart, "A5")


    # sales_cat_ws.sheet_state = "hidden"

    # ##############################################################

    # '''
    # Product Category	Revenue	% of Total Revenue
    # Coffee	$XXX,XXX	XX.X%
    # Bakery	$XXX,XXX	XX.X%
    # Tea	    $XXX,XXX	XX.X%
    # ...	    ...	        ...

    # '''
    # ##############################################################

    # # blank row
    # sales_analysis["A6"] = ""
    # sales_analysis.row_dimensions[6].height = 30

    # sales_analysis['A7'] = "Revenue Over Time"
    # sales_analysis['A7'].font = header_14
    # sales_analysis.row_dimensions[7].height = 36
    # '''Monthly Revenue Chart'''

    # # blank row
    # sales_analysis["A8"] = ""
    # sales_analysis.row_dimensions[8].height = 30

    # sales_analysis['A9'] = "Product Revenue Data"
    # sales_analysis['A9'].font = header_14
    # sales_analysis.row_dimensions[9].height = 36
    # '''
    # Rank	Product	Revenue
    # 1	    ...	    $...
    # 2	    ...	    $...
    # 3	    ...	    $...
    # '''
    # # blank row
    # sales_analysis["A10"] = ""
    # sales_analysis.row_dimensions[10].height = 30

    # # further investigation
    # sales_analysis['A11'] = "Further Investigation"
    # sales_analysis['A11'].font = header_14
    # sales_analysis.row_dimensions[11].height = 36
    # '''
    # Are high-volume products also high-revenue products?
    # Are monthly trends consistent across categories?'''


    # ######## THIRD SHEET - LOCATION PERFORMANCE #######
    # location_performance = wb.create_sheet(title="Location Performance")

    # # set columns width
    # location_performance.column_dimensions['A'].width = 20
    # location_performance.column_dimensions['B'].width = 20
    # location_performance.column_dimensions['C'].width = 20
    # location_performance.column_dimensions['D'].width = 20
    # location_performance.column_dimensions['E'].width = 20

    # # set header
    # location_performance['A1'] = "Location Performance"
    # location_performance['A1'].font = header_18
    # location_performance.row_dimensions[1].height = 60
    # location_performance.merge_cells('A1:E1')
    # location_performance['A1'].alignment = center_align


    # # set subheader
    # location_performance['A3'] = "Revenue performance and product mix across store locations, including differences by time of day."
    # location_performance['A3'].font = header_14
    # location_performance.row_dimensions[3].height = 36
    # location_performance.merge_cells('A3:E3')
    # location_performance['A3'].alignment = center_align

    # # blank row
    # location_performance["A4"] = ""
    # location_performance.row_dimensions[4].height = 30


    # location_performance['A4'] = "Product Mix by Location and TIme of Day"
    # location_performance['A4'].font = header_14
    # location_performance.row_dimensions[4].height = 36
    # '''Location × Time Period × Product Category Chart'''

    # # blank row
    # location_performance["A5"] = ""
    # location_performance.row_dimensions[5].height = 30

    # location_performance['A6'] = "Revenue by Location"
    # location_performance['A6'].font = header_14
    # location_performance.row_dimensions[6].height = 36
    # '''Revenue by Location Chart'''

    # # blank row
    # location_performance["A7"] = ""
    # location_performance.row_dimensions[7].height = 30

    # location_performance['A8'] = "Location Revenue Data"
    # location_performance['A8'].font = header_14
    # location_performance.row_dimensions[8].height = 36
    # '''
    # TOP LOCATION
    # Lower Manhattan

    # REVENUE
    # $XXX,XXX

    # SHARE OF TOTAL
    # XX.X%
    # '''

    # # blank row
    # location_performance["A9"] = ""
    # location_performance.row_dimensions[9].height = 30

    # # further investigation
    # location_performance['A10'] = "Further Investigation"
    # location_performance['A10'].font = header_14
    # location_performance.row_dimensions[10].height = 36

    # '''Do customers at different locations have significantly different product preferences?
    # Are certain products disproportionately strong at particular locations?
    # Does the product mix change more by location or by time of day?
    # Do lower-performing locations have the same product mix as higher-performing locations?
    # Are there products that perform particularly well during specific time periods at only one location?
    # '''


    # ######## FOURTH SHEET - Time ANALYSIS #######
    # time_analysis = wb.create_sheet(title="Time Analysis")

    # # set columns width
    # time_analysis.column_dimensions['A'].width = 20
    # time_analysis.column_dimensions['B'].width = 20
    # time_analysis.column_dimensions['C'].width = 20
    # time_analysis.column_dimensions['D'].width = 20
    # time_analysis.column_dimensions['E'].width = 20


    # # set header
    # time_analysis['A1'] = "Time Analysis"
    # time_analysis['A1'].font = header_18
    # time_analysis.row_dimensions[1].height = 60
    # time_analysis.merge_cells('A1:E1')
    # time_analysis['A1'].alignment = center_align

    # # set subheader
    # time_analysis['A3'] = "When are customers generating the most sales?"
    # time_analysis['A3'].font = header_14
    # time_analysis.row_dimensions[3].height = 36
    # time_analysis.merge_cells('A3:E3')
    # time_analysis['A3'].alignment = center_align

    # # blank row
    # time_analysis["A4"] = ""
    # time_analysis.row_dimensions[4].height = 30


    # time_analysis['A4'] = "Revenue by Hour"
    # time_analysis['A4'].font = header_14
    # time_analysis.row_dimensions[4].height = 36
    # '''Large Primary chart'''

    # # blank row
    # time_analysis["A5"] = ""
    # time_analysis.row_dimensions[5].height = 30

    # time_analysis['A6'] = "Revenue by Day"
    # time_analysis['A6'].font = header_14
    # time_analysis.row_dimensions[6].height = 36
    # '''day of the week chart'''

    # time_analysis['C6'] = "Revenue by Time Period"
    # time_analysis['C6'].font = header_14
    # time_analysis.row_dimensions[6].height = 36
    # '''"Time Period" chart'''

    # # blank row
    # time_analysis["A7"] = ""
    # time_analysis.row_dimensions[7].height = 30

    # time_analysis['A8'] = "product Category x Time Period"
    # time_analysis['A8'].font = header_14
    # time_analysis.row_dimensions[8].height = 36

    # '''
    # Category       Morning  Lunch  Afternoon  Evening 
    # Bakery          XX.X%    XX.X%    XX.X%      XX.X%
    # Coffee          XX.X%    XX.X%    XX.X%      XX.X%
    # '''
    # # blank row
    # time_analysis["A9"] = ""
    # time_analysis.row_dimensions[9].height = 30

    # # further investigation
    # time_analysis['A10'] = "Further Investigation"
    # time_analysis['A10'].font = header_14
    # time_analysis.row_dimensions[10].height = 36
    # '''
    # Does transaction volume follow revenue by hour?            │
    # Are weekday/weekend patterns substantially different?       │
    # Are particular categories especially dependent on time? 
    # '''

    # back to first sheet
    wb.active = kpi_sheet

    # Save the file
    report_name = (
    f"Performance Report - "
    f"{quarter} - {start_month} – {end_month} {year}.xlsx"
    )
    #------------------------------------
    print(f"Saving report: {report_name}")
    wb.save(report_name)

Saving report: Performance Report - 2023Q1 - January – March 2023.xlsx
Saving report: Performance Report - 2023Q2 - April – June 2023.xlsx
